# 01 - Exploración de datos electorales (SICEE)

Este notebook carga las series generadas por `scripts/build_series.py` y realiza un análisis exploratorio básico.

## Instrucciones para Google Colab
1. Sube los archivos CSV de `data/processed/series/` a la sesión de Colab (o monta Google Drive).
2. Ejecuta las celdas de arriba hacia abajo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, glob

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('pandas', pd.__version__, '| numpy', np.__version__)

## Cargar serie de Presidencia por entidad

In [ ]:
# En Colab cambia BASE al path donde subiste los CSV
BASE = '../data/processed/series'  # local
# BASE = '/content/drive/MyDrive/inteligencia-electoral-mexico/data/processed/series'  # Drive

df = pd.read_csv(os.path.join(BASE, 'serie_presidencia_entidad.csv'))
df['VOTOS'] = pd.to_numeric(df['VOTOS'], errors='coerce').fillna(0)
df['PARTICIPACION'] = df['TOTAL_VOTOS'] / df['LISTA_NOMINAL']
print('Filas:', len(df))
print('Años:', sorted(df['ANIO'].unique()))
df.head()

## Votación histórica por partido (nacional)

In [ ]:
nacional = df.groupby(['ANIO', 'PARTIDO_COALICION'])['VOTOS'].sum().reset_index()
pivot = nacional.pivot(index='ANIO', columns='PARTIDO_COALICION', values='VOTOS').fillna(0)
pivot.plot(kind='bar', stacked=True, colormap='tab20')
plt.title('Votos por partido/coalición en elecciones presidenciales (nacional)')
plt.ylabel('Votos')
plt.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.show()

## Mapa de calor: participación por entidad y año

In [ ]:
part = df.groupby(['ANIO', 'ENTIDAD'])['PARTICIPACION'].mean().reset_index()
pivot_part = part.pivot(index='ENTIDAD', columns='ANIO', values='PARTICIPACION')
sns.heatmap(pivot_part, cmap='YlOrRd', annot=False)
plt.title('Participación por entidad y año (Presidencia)')
plt.tight_layout()
plt.show()

## Exportar panel limpio para modelos

In [ ]:
panel = df.copy()
panel['VOTO_NORMALIZADO'] = panel['VOTOS'] / panel['TOTAL_VOTOS']
panel.to_csv('panel_presidencia_entidad.csv', index=False)
print('Guardado panel_presidencia_entidad.csv')